#### Optimizer 최적화기 옵티마이저
:  손실함수를 통계 계산한 기울기를 이용하여 신경망의 가중치와 편향을 업데이트하는 알고리즘

- 신경망 학습의 순서
    0. 입력데이터 전처리
    1. 입력데이터를 모델에 전달
    2. 모델이 예측값을 계산
    3. 손실함수로 예측값과 검증값의 차이를 계산
    4. loss.backward()로 각 파라미터의 gradient를 계산
    5. optimizer.step()으로 파라미터 업데이트

- 주요 구성 요소

| 구성 요소 | 역할 |
| --- | --- |
| 손실함수 | 예측값과 정답의 차이 |
| Gradient | 파라미터를 조금 변경했을 손실이 변하는 방향과 정도 |
| Optimizer | Gradient를 이용하여 파라미터 업데이트 |
| Learning Rate | 한번 업데이트할 때 변경하는 크기 |

- Optimizer 범례 :
Gradient Descent
Adam
SGD
momentum
RMSprop

#### 1. Gradient Descent 경사하강법
: 손실함수가 작아지는 방향으로 파라미터를 조금씩 변경하는 방법

- Gradient가 양수이면 파라미터를 감소시키고, 음수이면 파라미터를 증가시키는 방향으로 업데이트

- 손실이 증가하는 방향을 나타내므로 gradient의 반대 방향으로 파라미터를 이동시켜 손실을 최소화

- 용어설명
    - 파라미터(parameters): 모델이 학습을 통해 조정하는 모든 값
    - 가중치(weight) : 입력 데이터의 출력값에 대한 영향력을 조절하는 파라미터
    - 편향(bias) : 입력과 별개로 출력값을 조정하는 파라미터

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
# 간단한 파라미터와 손실 함수
w = torch.tensor([2.], requires_grad= True)
target = torch.tensor([10.])

pred = w * 3
loss = (pred - target) ** 2 # 입력값이 하나이므로 mean 생략

loss.backward()

print('prediction:', pred)
print('loss:', loss)
print('w의 grad:', w.grad)
print('현재 w:', w.item())


prediction: tensor([6.], grad_fn=<MulBackward0>)
loss: tensor([16.], grad_fn=<PowBackward0>)
w의 grad: tensor([-24.])
현재 w: 2.0


In [3]:
# 새로운 가중치 계산
w_old = w.item()
gradient = w.grad.item() # 손실함수의 기울기 = dL / dw
learning_rate = 0.1

w_new = w_old - learning_rate * gradient

print('과거 w:', w_old)
print('gradient:', gradient)
print('learning rate:', learning_rate)
print('업데이트 후 w:', w_new)

과거 w: 2.0
gradient: -24.0
learning rate: 0.1
업데이트 후 w: 4.4


In [4]:
w = torch.tensor([2.], requires_grad= True)
target = torch.tensor([10.])

learning_rate = 0.1

for step in range(10):
    # 예측 및 손실 계산
    pred = w * 3 
    loss = (pred - target) ** 2
    
    # gradient 계산
    loss.backward()
    print(f'====================={step+ 1} 차 학습====================')
    print('prediction:', pred.item())
    print('loss:', loss.item())
    print('gradient:', w.grad.item())
    print('업데이트 전 w :', w.item())

    # 파라미터 업데이트
    with torch.no_grad():    # 추적금지 요청 중요
        w -= learning_rate * w.grad
    
    print('업데이트 후 w:', w.item())

    # 다음 학습을 위해 gradient 초기화
    w.grad.zero_()



=====================1 차 학습====================
prediction: 6.0
loss: 16.0
gradient: -24.0
업데이트 전 w : 2.0
업데이트 후 w: 4.400000095367432
=====================2 차 학습====================
prediction: 13.200000762939453
loss: 10.240004539489746
gradient: 19.20000457763672
업데이트 전 w : 4.400000095367432
업데이트 후 w: 2.479999542236328
=====================3 차 학습====================
prediction: 7.439998626708984
loss: 6.553606986999512
gradient: -15.360008239746094
업데이트 전 w : 2.479999542236328
업데이트 후 w: 4.016000270843506
=====================4 차 학습====================
prediction: 12.04800033569336
loss: 4.194305419921875
gradient: 12.288002014160156
업데이트 전 w : 4.016000270843506
업데이트 후 w: 2.7871999740600586
=====================5 차 학습====================
prediction: 8.361599922180176
loss: 2.684354782104492
gradient: -9.830400466918945
업데이트 전 w : 2.7871999740600586
업데이트 후 w: 3.770240068435669
=====================6 차 학습====================
prediction: 11.310720443725586
loss: 1.717988133430481
gradien

#### 2. Learning Rate 학습률
: 파라미터를 한 번 업데이트 할 때 얼마나 크게 변경할 것인지를 결정하는 값

**Learning Rate가 너무 작은 경우**
   - 파라미터의 변경크기가 지나치게 작아 손실을 줄이는 속도가 더딤
   - 학습이 필요로하는 반복 횟수가 지나치게 커짐
   - 학습곡선이 축에 거의 붙어있음

**Learning Rate가 너무 큰 경우**
   - 파라미터의 변경크기가 지나치게 커, 최적값을 지나치는 문제
   - 학습이 불안정하거나 발산할 수도 있음
   - 학습곡선이 평행선을 그리거나 줄어들지 않고 커질 수 있음<br>
   
   
**하이퍼파라미터**(사용자가 넣음): 모델과 데이터에 맞게 조정해야하는 중요한 파라미터

| Optimizer | 일반적인 시작 학습률 |
| --- | --- |
| 경사하강법, SGB | lr= 0.01 |
| SGD + momentum | lr = 0.01 ~ 0.1 |
| Adam | 0.001 |
| AdamW | 0.001 |
| RMSprop | 0.001 |

In [5]:
# gradient가 동일 할 때 Learning Rate에 따른 업데이트 크기 비교
w = 5.0
gradient = 2.0

for lr in [0.001, 0.1, 0.5] :
    update_w = w - (lr * gradient)
    print(f'learning rate = {lr} : 업데이트 후 w  {update_w}')

learning rate = 0.001 : 업데이트 후 w  4.998
learning rate = 0.1 : 업데이트 후 w  4.8
learning rate = 0.5 : 업데이트 후 w  4.0


#### 3. SGD(stochastic Gradient Descent)
: gradient를 이용하여 파라미터를 업데이트하는 기본적인 최적화 알고리즘

- 사용형식
```python
torch.optim.SGD(model.parameter(), lr=0.01)
```

- 특징
    - 구조가 단순하고 이해하기 쉬움
    - Learning Rate설정에 영향을 많이 받음
    - momentum옵션을 사용하여 이전 업데이트 방향을 반영할 수 있음

- 기본값은  momentum을 사용하지 않음(별도의 지정 필요)

In [6]:
model = nn.Linear(1, 1)

optimizer = optim.SGD(model.parameters(), lr=0.01)

print('Optimizer :', optimizer)
print('초기 weight :', model.weight)
print('초기 bias :', model.bias)

Optimizer : SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)
초기 weight : Parameter containing:
tensor([[0.6841]], requires_grad=True)
초기 bias : Parameter containing:
tensor([0.5976], requires_grad=True)


#### 4. Optimizer.step()
: backward()로 계산한 gradient값으로 optimizer에서 update한 파라미터 값을 model에 넣는 method

- 일반적인 순서

```python
optimizer.zero_grad()   # 이전 gradient 제거

pred = model(x)     # 예측값 계산
loss = loss_fn(pred, target)    # 손실 계산

loss.backward()     # gradient 계산
optimizer.step()    # 파라미터 업데이트
```
**중요!**
optimizer.step()을 호출하기 전에 반드시 loss.backward()실행할 것

In [11]:
w = torch.nn.Parameter(torch.tensor([5.]))
optimizer = optim.SGD([w], lr = 0.1)

target = torch.tensor([3.])

print('업데이트 전 w', w.item())
print('업데이트 전 loss', ((w-target)**2).item())
optimizer.zero_grad()
loss = (w - target)**2
loss.backward()

print('gradient ', w.grad.item())

optimizer.step()

print('업데이트 후 w', w.item())
print('업데이트 후 loss', ((w-target)**2).item())

업데이트 전 w 5.0
업데이트 전 loss 4.0
gradient  4.0
업데이트 후 w 4.599999904632568
업데이트 후 loss 2.559999704360962


In [15]:
x = torch.tensor([[1.0],[2.],[3.],[4.]])
target = torch.tensor([[5.],[8.],[11.],[14.]])

model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr= 0.01)

for epoch in range(100):
    optimizer.zero_grad()

    pred = model(x)
    loss = loss_fn(pred, target)
    
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f'Epoch {epoch:3d} | loss {loss.item():4f}')

print('학습 후 weight', model.weight.item())
print('학습 후 bias', model.bias.item())


Epoch   0 | loss 107.768097
Epoch  20 | loss 0.131983
Epoch  40 | loss 0.053170
Epoch  60 | loss 0.047118
Epoch  80 | loss 0.041793
학습 후 weight 3.16023850440979
학습 후 bias 1.5288801193237305


#### 5. Adam(Adaptive Moment Estimation)
: 학습과정에서 Gradient의 평균적인 방향과 크기를 함께 고려하여 파리미터를 업데이트하는 최적화 알고리즘

[사용방법]
```python
torch.optim.Adam(model.parameters(), lr= 0.001)
```
- Adam의 주요 특징
    - 각 파라미터의 Gradient를 바탕으로 업데이트 크기를 적응형 방식으로 조절
    - Gradient의 1차 모멘트(평균)와 2차 모멘트(제곱의 평균)을 이용

    - 일반적으로 기본 learning Rate로 0.001을 자주 사용함
    - 다양한 신경망 학습에 널리 사용됨

- 기본 개념
    - 1차 모멘트 : Gradient의 이동평균
        - Momentum: Stores an exponentially decaying average of past gradients (first-order moment/mean) to help accelerate training along relevant directions and reduce oscillation.
    - 2차 모멘트 : Gradient 제곱의 이동평균
        - RMSprop: Stores an exponentially decaying average of past squared gradients (second-order moment/variance) to scale the learning rate adaptively for each parameter.

In [ ]:
model = nn.Linear(1,1)

optimizer = optim.Adam(model.parameters(), lr=0.001)

print('Optimizer', optimizer)
print('초기 weight', model.weight)
print('초기 bias', model.bias)

Optimizer Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
초기 weight Parameter containing:
tensor([[0.8534]], requires_grad=True)
초기 bias Parameter containing:
tensor([0.9139], requires_grad=True)


#### SGD 와 Adam 비교

| 구분 | SGD | Adam |
| -- | -- | -- | 
| 기본방식 | Gradient와 Learning Rate로 업데이트 | Gradient에 1,2차 모멘트 활용 |
| 학습률 | 사용자가 설정 | 적응적 업데이트 |
| 특징 | 단순하고 직관적 | 다양한 문제에서 잘 진행되는 경우가 많음 |
| 대표적인 시작값 | 0.01 | 0.001 |

**적용 대상**
- SGD :

In [ ]:
# 동일한 데이터로 SGD 와 Adam 학습결과 비교
torch.manual_seed(42)

x = torch.tensor([[1.],[2.],[3.],[4.]])
target = 4*x -1 # weight : 4, bias: 1

model_sgd = nn.Linear(1,1)
model_adam = nn.Linear(1,1)

# 두 모델의 weight 와 bias 를 일치시키는 법
model_adam.load_state_dict(model_sgd.state_dict())

loss_fn = nn.MSELoss()
optimizer_sgd = optim.SGD(model_sgd.parameters(), lr= 0.01)
optimizer_adam = optim.Adam(model_adam.parameters(), lr= 0.01)

for epoch in range(200):
    # SGD 학습
    optimizer_adam.zero_grad()

    pred_sgd = model_sgd(x)
    loss_sgd = loss_fn(pred_sgd, target)
    loss_sgd.backward()
    optimizer_sgd.step()

    # Adam 학습
    optimizer_adam.zero_grad()

    pred_adam = model_adam(x)
    loss_adam = loss_fn(pred_adam, target)
    loss_adam.backward()
    optimizer_adam.step()

    if epoch % 50 == 0 :
        print(f'Epoch {epoch}')
        print(f'SGD Loss {loss_sgd.item():6f}')
        print(f'Adam Loss {loss_adam.item():6f}')
        print('-'*30)

print('SGD 최종 Loss', loss_fn(pred_sgd, target).item())
print('Adam 최종 Loss', loss_fn(pred_adam, target).item())

Epoch 0
SGD Loss 52.255905
Adam Loss 52.255905
------------------------------
Epoch 50
SGD Loss 6.955769
Adam Loss 30.350632
------------------------------
Epoch 100
SGD Loss 85.465927
Adam Loss 16.560976
------------------------------
Epoch 150
SGD Loss 27.932617
Adam Loss 8.831192
------------------------------
SGD 최종 Loss 51.65553283691406
Adam 최종 Loss 5.002765655517578


In [ ]:
# Learning Rate 에 따른 학습 비교

x = torch.tensor([[1.],[2.],[3.],[4.]])
target = 4*x -1 # weight : 4, bias: 1

initial_model = nn.Linear(1,1)
initial_state = initial_model.state_dict()

learning_rates = [0.001, 0.03, 0.5]

for lr in learning_rates:
    model = nn.Linear(1,1)
    model.load_state_dict(initial_state)

    loss_fn = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)

    losses = []
    for epoch in range(30):
        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, target)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    print(f'Learning rate = {lr:<5} | 처음 loss = {losses[0]:.4f} | 마지막 loss = {losses[-1]:4f}')

Learning rate = 0.001 | 처음 loss = 117.8989 | 마지막 loss = 45.178421
Learning rate = 0.03  | 처음 loss = 117.8989 | 마지막 loss = 0.770660
Learning rate = 0.5   | 처음 loss = 117.8989 | 마지막 loss =  inf


#### Epoch 와 반복학습
: Epoch는 학습 데이터 전체를 한번 학습한 단위 ( 주기 ) <br><br>

예를 들어 학습 데이터가 100개이고, 전체 데이터를 한 번 학습했다면 1 Epoch 기록<br><br>

- 학습을 거듭할 수록 loss는 계속 감소하지만, 그것만으로 성능이 뛰어나다고 말하기는 어려움
    - 과적합(overfitting)이 발생할 수 있음
    - 검증 데이터를 입력했을 때 loss가 충분히 감소하지 않았다면, 예측을 제대로 못하고 있다는 의미

In [25]:
torch.manual_seed(0)

x = torch.linspace(-2,2,20).reshape(-1,1) # size = ([20,1])
target = 2 * x + 3

model = nn.Linear(1,1)
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr= 0.05)

for epoch in range(101):
    optimizer.zero_grad()
    pred = model(x)
    loss = loss_fn(pred, target)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0 :
        print(f'Epoch {epoch:4d}, Loss {loss.item():4f}')

Epoch    0, Loss 12.008061
Epoch   20, Loss 3.849823
Epoch   40, Loss 0.660271
Epoch   60, Loss 0.048689
Epoch   80, Loss 0.003202
Epoch  100, Loss 0.000639
